In [1]:
import pandas as pd
import numpy as np
import csv
import hazm
import re
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
import pingouin as pg  

C:\Users\armin\AppData\Local\Temp\ipykernel_7616\3854743248.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


# Reading file

In [ ]:

# خواندن فایل‌های ارزیابی انسانی و مدل
human = pd.read_csv('human file.csv') # Human sentences
model = pd.read_csv('model file.csv') # Model sentences

# Lexical Idea

In [ ]:
# ======== hazm setup ========
normalizer = hazm.Normalizer()
lemmatizer = hazm.Lemmatizer()
tokenizer = hazm.WordTokenizer()

# ======== stopwords ========
stopwords = set(hazm.stopwords_list())

# ======== baseline: unique lemmas ========
def baseline_idea_count(text):
    if not isinstance(text, str) or text.strip() == "":
        return 0, 0
    
    text = normalizer.normalize(text)
    tokens = tokenizer.tokenize(text)

    # remove stopwords
    tokens = [t for t in tokens if t not in stopwords]

    # keep alphabetic only
    lemmas = [lemmatizer.lemmatize(t) for t in tokens if t.isalpha()]

    unique_lemmas = len(set(lemmas))
    total_tokens = len(lemmas)
    
    return unique_lemmas


# ======== query-wise metrics ========
def query_wise_idea_counts(df):
    results = []
    for query, group in df.groupby("Query"):
        baseline_counts = []
        baseline_per100 = []
        # robust_counts = []
        for s in group["Sentence"]:
            u = baseline_idea_count(s)
            baseline_counts.append(u)
        results.append({
            "Topic": query,
            "Lexical Idea Count (avg)": sum(baseline_counts) / len(baseline_counts),
            "count_sentences": len(group)
        })
    return pd.DataFrame(results)

# Human

In [ ]:
human_majority = pd.read_csv('file.csv') # Human majority
model_majority = pd.read_csv('file_model.csv') # Model majority

In [5]:
def sentence_wise_idea_scores(df):
    results = []
    for idx, row in df.iterrows():
        text = row["Sentence"]

        # baseline
        u = baseline_idea_count(text)

        results.append({
            "Sentence": text,
            "Lexical_Idea_Count": u,
        })
    return pd.DataFrame(results)


In [6]:
human_idea = sentence_wise_idea_scores(human)
human_idea.drop(index=[100],inplace=True)

In [ ]:
fluency = human_majority['Fluency']
lexical_idea_human = human_idea['Lexical_Idea_Count']

# Model

In [ ]:
model_idea = sentence_wise_idea_scores(model)

In [ ]:
fluency_model = model_majority['Fluency']
lexical_idea_model = model_idea['Lexical_Idea_Count']

# Get Correlation

In [ ]:
from scipy.stats import spearmanr

rho, p = spearmanr(x, y)

print("Spearman correlation:", rho)
print("p-value:", p)


# Save File

lexical_idea_human.to_csv("file.csv", index=False)

In [ ]:
lexical_idea_model.to_csv("file.csv", index=False)